# Prompt Evaluation Analysis

This notebook loads the results from the latest MLflow evaluation run and calculates metrics such as overall accuracy and failure rates per scorer.


In [1]:
import os
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

# Initialize MLflow client
db_path = os.path.abspath(os.path.join(os.getcwd(), "..", "mlflow.db")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath("mlflow.db")
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", f"sqlite:///{db_path}")
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()

experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "customer-service-bot")
experiment = client.get_experiment_by_name(experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")


Experiment ID: 2


In [2]:
# Fetch the latest full evaluation run
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'prompt-eval-full%'",
    order_by=["start_time DESC"],
    max_results=1
)

if not runs:
    print("No full evaluation runs found.")
else:
    latest_run = runs[0]
    print(f"Latest Run ID: {latest_run.info.run_id}")
    print(f"Run Name: {latest_run.data.tags.get('mlflow.runName')}")


Latest Run ID: 372de732e71649debbf6da842b5effd4
Run Name: prompt-eval-full-20260716-175009


In [3]:
import os

# Load the local evaluation_results.csv
csv_path = os.path.abspath(os.path.join(os.getcwd(), "..", "evaluation_results.csv")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath("evaluation_results.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} records from {csv_path}.")
else:
    print(f"No local evaluation_results.csv found at {csv_path}.")


Loaded 23 records from /home/phen/projects/customer-service-scheduling-bot/evaluation_results.csv.


In [4]:
# Calculate failure rates per scorer
# The columns we care about are the value columns for each scorer
scorer_cols = [col for col in df.columns if col.endswith("/value")]

print("--- Scorer Success Rates ---")
for col in scorer_cols:
    scorer_name = col.split("/")[0]
    # Handle cases where scorer might have failed to run (nulls)
    valid_responses = df[col].dropna()
    if len(valid_responses) == 0:
        print(f"{scorer_name}: No valid responses")
        continue
        
    success_count = (valid_responses.str.lower() == "yes").sum()
    total_count = len(valid_responses)
    success_rate = success_count / total_count
    
    print(f"{scorer_name:20s}: {success_rate:.1%} ({success_count}/{total_count})")


--- Scorer Success Rates ---
ResponseQuality     : 87.0% (20/23)
DomainContainment   : 47.8% (11/23)
HallucinationCheck  : 100.0% (22/22)
RoutingAccuracy     : 100.0% (23/23)
ToolSelection       : 50.0% (10/20)
